# When ML Fails: Paddy Yield Regression

This notebook is the notebook deliverable for the mini-project. The full reproducible implementation lives in `src/run_experiments.py`; this notebook documents and runs the same experiment.

## Research Question

Does a Random Forest trained to predict total paddy yield rely on cultivated-area and raw input-quantity features as a scale shortcut, producing excellent random-split performance but failing when evaluated on unseen cultivated-area groups?

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

DATA_PATH = Path(r'C:\Users\Admin\OneDrive\Documents\paddydataset.csv')
OUTPUT_DIR = PROJECT_ROOT / 'results'
OUTPUT_DIR.mkdir(exist_ok=True)

DATA_PATH

In [ ]:
import pandas as pd
from run_experiments import clean_columns, summarize_by_hectare

df = clean_columns(pd.read_csv(DATA_PATH))
df.shape, df.head()

In [ ]:
hectare_summary = summarize_by_hectare(df)
hectare_summary

## Reference Model and Failure Test

The reference model predicts total yield directly. The controlled out-of-distribution test leaves one `Hectares` group out at a time.

In [ ]:
from run_experiments import run_random_split, run_leave_one_hectare_out, run_ablation

random_split = run_random_split(df, pipeline='both')
leave_one_hectare = run_leave_one_hectare_out(df, pipeline='both')
ablation = run_ablation(df)

random_split.groupby('model')[['MAE', 'RMSE', 'R2']].agg(['mean', 'std']).round(3)

In [ ]:
leave_one_hectare.groupby('model')[['MAE', 'RMSE', 'R2']].mean().round(3)

In [ ]:
ablation[['split', 'MAE', 'RMSE', 'R2']].round(3)

## Figures

These are generated by the same plotting functions used by the script.

In [ ]:
from run_experiments import plot_target_scale, plot_failure

plot_target_scale(hectare_summary, OUTPUT_DIR)
plot_failure(leave_one_hectare, OUTPUT_DIR)

OUTPUT_DIR / 'target_scale_by_hectare.png', OUTPUT_DIR / 'leave_one_hectare_mae.png'

## Conclusion

The Random Forest appears excellent on random splits, but it fails under leave-one-hectare-out evaluation. The per-hectare correction changes the representation so the model learns productivity rather than raw field scale.